# RavenStack — Customer Churn Prediction
## Notebook 01: Exploratory Data Analysis (EDA)

**Tujuan notebook ini:**
1. Distribusi target & class imbalance
2. Distribusi & outlier tiap fitur
3. Multikolinearitas antar fitur
4. Korelasi fitur terhadap target (churn)
5. Justifikasi pemilihan model tree-based vs linear

---
### Catatan Asumsi Model

| Asumsi | Logistic Regression | XGBoost / Random Forest |
|---|---|---|
| Tidak ada multikolinearitas | ✅ Wajib | ❌ Tidak perlu |
| Hubungan linear dengan target | ✅ Wajib | ❌ Tidak perlu |
| Normalitas fitur | ⚠️ Disarankan | ❌ Tidak perlu |
| Independensi observasi | ✅ Wajib | ✅ Wajib |

> Karena kita memilih **XGBoost & Random Forest**, asumsi multikolinearitas
> dan linearitas tidak wajib. Namun tetap dicek untuk kualitas data.

---
## 0. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import warnings, os
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')
os.makedirs('../reports', exist_ok=True)

C_CHURN    = '#E24B4A'
C_NO_CHURN = '#1D9E75'
C_NEUTRAL  = '#7F77DD'

print('✅ Library siap')

---
## 1. Load Data

In [ ]:
df = pd.read_csv('../data/processed/ravenstack_features_for_modeling.csv')
bool_cols = df.select_dtypes(include='bool').columns.tolist()
df[bool_cols] = df[bool_cols].astype(int)

X = df.drop(columns=['target'])
y = df['target']

print(f'Shape   : {df.shape}')
print(f'Fitur   : {X.shape[1]}')
print(f'Churn   : {y.sum()} ({y.mean()*100:.1f}%)')
print(f'No churn: {(y==0).sum()} ({(y==0).mean()*100:.1f}%)')
print(f'Nulls   : {df.isnull().sum().sum()}')

---
## 2. Class Imbalance & Target Distribution

> **Relevansi:** Menentukan metrik evaluasi yang tepat.
> Data imbalanced → **PR-AUC lebih relevan dari AUC-ROC**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
counts = y.value_counts()
bars = axes[0].bar(['Tidak Churn (0)', 'Churn (1)'],
                    counts.values,
                    color=[C_NO_CHURN, C_CHURN], width=0.5, edgecolor='white')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+3,
                 f'{val}\n({val/len(y)*100:.1f}%)', ha='center', fontweight='bold')
axes[0].set_title('Distribusi Target Variable', fontweight='bold')
axes[0].set_ylabel('Jumlah')

# Imbalance ratio
ratio = counts[0] / counts[1]
axes[1].barh(['Imbalance Ratio'], [ratio], color=C_NEUTRAL, height=0.4)
axes[1].axvline(x=1, color='gray', linestyle='--', alpha=0.5, label='Balanced (ratio=1)')
axes[1].text(ratio+0.1, 0, f'{ratio:.2f}:1', va='center', fontweight='bold', fontsize=14)
axes[1].set_title('Class Imbalance Ratio', fontweight='bold')
axes[1].set_xlabel('Tidak Churn : Churn')
axes[1].legend()

plt.suptitle('⚠️ Data Imbalanced → Gunakan SMOTE + PR-AUC sebagai metrik utama',
              fontsize=11, color=C_CHURN, y=1.02)
plt.tight_layout()
plt.savefig('../reports/eda_01_class_imbalance.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Imbalance ratio : {ratio:.2f}:1')
print(f'Kesimpulan      : Data IMBALANCED → AUC-ROC bisa menyesatkan')
print(f'Solusi          : Pakai SMOTE saat training + evaluasi dengan PR-AUC & Recall')

---
## 3. Distribusi Fitur & Outlier

> **Relevansi untuk tree-based model:** Outlier ekstrem tidak terlalu bermasalah
> karena tree mempartisi berdasarkan threshold, bukan jarak.
> Namun outlier ekstrem bisa jadi sinyal data quality issue.

In [ ]:
# Pilih fitur numerik utama (non one-hot)
key_features = [
    'tenure_days', 'seats', 'total_mrr', 'avg_mrr',
    'days_since_last_usage', 'unique_features_used',
    'total_tickets', 'avg_satisfaction_score',
    'escalation_rate', 'error_rate',
    'feature_breadth_ratio', 'usage_density'
]
key_features = [f for f in key_features if f in X.columns]

fig, axes = plt.subplots(3, 4, figsize=(16, 11))
axes = axes.flatten()

for i, col in enumerate(key_features):
    ax = axes[i]
    # Histogram churn vs tidak
    ax.hist(X[col][y==0], bins=25, alpha=0.6, color=C_NO_CHURN,
             label='Tidak Churn', density=True)
    ax.hist(X[col][y==1], bins=25, alpha=0.6, color=C_CHURN,
             label='Churn', density=True)
    
    skew_val = X[col].skew()
    ax.set_title(f'{col}\nskew={skew_val:.2f}', fontsize=9, fontweight='bold')
    ax.legend(fontsize=7)
    ax.tick_params(labelsize=7)

plt.suptitle('Distribusi Fitur Utama: Churn vs Tidak Churn\n'
              '(overlap besar = fitur kurang diskriminatif)',
              fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../reports/eda_02_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Tersimpan: reports/eda_02_feature_distributions.png')

In [ ]:
# Skewness analysis
num_cols = X.select_dtypes(include='number').columns.tolist()
skew_df = X[num_cols].skew().abs().sort_values(ascending=False)

print('Distribusi level skewness:')
print(f'  Simetris     (|skew| < 0.5)  : {(skew_df < 0.5).sum()} fitur')
print(f'  Moderate     (0.5–1.0)       : {((skew_df >= 0.5) & (skew_df < 1.0)).sum()} fitur')
print(f'  Highly skewed (|skew| > 1.0) : {(skew_df >= 1.0).sum()} fitur')
print()
print('Top 10 fitur paling skewed:')
print(skew_df.head(10).to_string())
print()
print('Catatan untuk tree-based model:')
print('  Skewness TIDAK masalah untuk XGBoost & Random Forest.')
print('  Tree mempartisi data berdasarkan threshold, bukan distribusi.')
print('  Transformasi log/sqrt tidak diperlukan.')

In [ ]:
# Boxplot outlier — fitur kunci
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

outlier_features = ['total_mrr', 'tenure_days', 'total_tickets',
                     'avg_satisfaction_score', 'days_since_last_usage', 'escalation_rate']
outlier_features = [f for f in outlier_features if f in X.columns]

for i, col in enumerate(outlier_features):
    data_plot = [X[col][y==0].dropna(), X[col][y==1].dropna()]
    bp = axes[i].boxplot(data_plot, labels=['Tidak Churn', 'Churn'],
                          patch_artist=True, notch=False)
    bp['boxes'][0].set_facecolor(C_NO_CHURN)
    bp['boxes'][1].set_facecolor(C_CHURN)
    for patch in bp['boxes']:
        patch.set_alpha(0.7)
    axes[i].set_title(col, fontweight='bold')
    axes[i].tick_params(labelsize=9)

plt.suptitle('Boxplot Fitur Kunci: Deteksi Outlier & Perbedaan Distribusi Churn vs Tidak',
              fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../reports/eda_03_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Tersimpan: reports/eda_03_boxplots.png')

---
## 4. Multikolinearitas

> **Relevansi:**
> - Untuk **Logistic Regression**: multikolinearitas tinggi → koefisien tidak stabil → WAJIB diatasi
> - Untuk **XGBoost/RF**: tidak mempengaruhi performa, tapi fitur yang hampir identik
>   membuang resource dan membingungkan interpretasi SHAP
>
> **Temuan awal:** Ada **35 pasangan fitur** dengan korelasi > 0.85, termasuk
> beberapa yang korelasi = 1.0 (hampir identik!)

In [ ]:
# Hitung correlation matrix — fitur numerik utama saja
corr_features = [
    'tenure_days', 'seats', 'total_mrr', 'avg_mrr', 'total_arr',
    'total_subscriptions', 'active_subscriptions', 'sub_churn_ratio',
    'total_usage_events', 'total_usage_count', 'usage_days',
    'unique_features_used', 'feature_breadth_ratio',
    'total_usage_duration_secs', 'error_rate', 'avg_error_count',
    'days_since_last_usage', 'usage_density',
    'total_tickets', 'avg_satisfaction_score', 'escalation_rate',
    'avg_resolution_hours', 'pct_urgent'
]
corr_features = [f for f in corr_features if f in X.columns]

corr_matrix = X[corr_features].corr()

fig, ax = plt.subplots(figsize=(16, 13))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, vmin=-1, vmax=1,
    ax=ax, linewidths=0.5, annot_kws={'size': 7},
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Correlation Matrix — Fitur Utama\n'
              '(merah=korelasi positif tinggi, hijau=negatif, putih=tidak berkorelasi)',
              fontweight='bold', fontsize=12)
ax.tick_params(labelsize=8)
plt.tight_layout()
plt.savefig('../reports/eda_04_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Tersimpan: reports/eda_04_correlation_matrix.png')

In [ ]:
# Identifikasi pasangan korelasi sangat tinggi
corr_full = X.select_dtypes(include='number').corr().abs()
upper = corr_full.where(np.triu(np.ones(corr_full.shape), k=1).astype(bool))

high_corr = [
    {'Fitur A': col, 'Fitur B': row, 'Korelasi': upper.loc[row, col]}
    for col in upper.columns
    for row in upper.index
    if pd.notna(upper.loc[row, col]) and upper.loc[row, col] > 0.85
]
high_corr_df = pd.DataFrame(high_corr).sort_values('Korelasi', ascending=False)

print(f'Pasangan fitur dengan korelasi > 0.85: {len(high_corr_df)}')
print()
print(high_corr_df.head(15).to_string(index=False))
print()
print('Fitur yang hampir IDENTIK (korelasi ≈ 1.0) → kandidat untuk di-drop:')
identical = high_corr_df[high_corr_df['Korelasi'] > 0.99]
print(identical.to_string(index=False))

In [ ]:
# Rekomendasi fitur yang bisa di-drop
# (fitur B adalah duplikat dari fitur A)
drop_candidates = {
    'total_arr'            : 'Identik dengan total_mrr (korelasi=1.0)',
    'error_rate'           : 'Identik dengan avg_error_count (korelasi=1.0)',
    'feature_breadth_ratio': 'Identik dengan unique_features_used (korelasi=1.0)',
    'total_usage_count'    : 'Hampir identik dengan total_usage_events (>0.99)',
    'usage_days'           : 'Hampir identik dengan total_usage_events (>0.99)',
    'total_usage_duration_secs': 'Sangat berkorelasi dengan usage events (>0.96)',
}

print('=' * 60)
print('REKOMENDASI: Fitur kandidat drop (multikolinearitas ekstrem)')
print('=' * 60)
for feat, alasan in drop_candidates.items():
    status = '✅ ada' if feat in X.columns else '❌ tidak ada'
    print(f'  {feat:35} | {alasan}')
print()
print('Catatan:')
print('  Untuk XGBoost/RF: drop ini OPSIONAL (tidak wajib)')
print('  Untuk Logistic Regression: drop ini WAJIB')
print('  Untuk SHAP interpretasi: sebaiknya di-drop agar tidak membingungkan')

---
## 5. Korelasi Fitur terhadap Target (Churn)

> Fitur mana yang paling berkorelasi dengan churn?
> Ini memberi sinyal awal fitur mana yang paling prediktif.

In [ ]:
# Point-biserial correlation (fitur numerik vs target biner)
num_cols = X.select_dtypes(include='number').columns.tolist()

corr_target = []
for col in num_cols:
    corr_val, pval = stats.pointbiserialr(y, X[col])
    corr_target.append({'feature': col, 'correlation': corr_val, 'pvalue': pval})

corr_target_df = pd.DataFrame(corr_target)
corr_target_df['abs_corr'] = corr_target_df['correlation'].abs()
corr_target_df['significant'] = corr_target_df['pvalue'] < 0.05
corr_target_df = corr_target_df.sort_values('abs_corr', ascending=False)

# Visualisasi top 20
top20 = corr_target_df.head(20)

fig, ax = plt.subplots(figsize=(11, 8))
colors = [C_CHURN if v > 0 else C_NO_CHURN for v in top20['correlation']]
bars = ax.barh(top20['feature'][::-1], top20['correlation'][::-1],
                color=colors[::-1], edgecolor='white', alpha=0.85)
ax.axvline(x=0, color='gray', linewidth=1)
ax.set_xlabel('Point-Biserial Correlation dengan Target (Churn)')
ax.set_title('Top 20 Fitur — Korelasi terhadap Churn\n'
              '(merah=positif→churn, hijau=negatif→tidak churn)',
              fontweight='bold')

# Tandai yang signifikan
sig_map = dict(zip(top20['feature'], top20['significant']))
for bar, feat in zip(bars[::-1], top20['feature'][::-1]):
    if sig_map[feat]:
        ax.text(bar.get_width() + 0.002 if bar.get_width() > 0 else bar.get_width() - 0.002,
                bar.get_y() + bar.get_height()/2,
                '*', va='center', fontsize=12, color='black')

ax.text(0.98, 0.02, '* = signifikan (p < 0.05)', transform=ax.transAxes,
         ha='right', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('../reports/eda_05_feature_correlation_target.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Tersimpan: reports/eda_05_feature_correlation_target.png')
print()
print('Top 10 fitur paling berkorelasi dengan churn:')
print(corr_target_df[['feature','correlation','pvalue','significant']].head(10).to_string(index=False))

---
## 6. Linearitas — Apakah Hubungan Fitur-Target Linear?

> **Relevansi:**
> Logistic Regression mengasumsikan hubungan **linear** antara fitur dan log-odds target.
> Kalau tidak linear → LR performa buruk → justifikasi pakai XGBoost/RF.

In [ ]:
# Cek linearitas dengan binned mean plot
# Kalau hubungan linear → garis hampir lurus
# Kalau tidak linear → garis melengkung → tree-based lebih cocok

check_features = [
    'tenure_days', 'days_since_last_usage',
    'avg_satisfaction_score', 'total_mrr',
    'feature_breadth_ratio', 'escalation_rate'
]
check_features = [f for f in check_features if f in X.columns]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(check_features):
    ax = axes[i]
    temp = pd.DataFrame({'feature': X[col], 'target': y})
    temp = temp.dropna()
    
    # Bagi jadi 10 bin, hitung rata-rata churn per bin
    temp['bin'] = pd.qcut(temp['feature'], q=10, duplicates='drop')
    binned = temp.groupby('bin', observed=True)['target'].agg(['mean','count']).reset_index()
    binned['bin_mid'] = binned['bin'].apply(lambda x: x.mid)
    
    ax.plot(binned['bin_mid'], binned['mean'],
             'o-', color=C_NEUTRAL, linewidth=2, markersize=6)
    ax.fill_between(binned['bin_mid'], binned['mean'], alpha=0.15, color=C_NEUTRAL)
    
    # Garis linear sebagai referensi
    z = np.polyfit(binned['bin_mid'], binned['mean'], 1)
    p = np.poly1d(z)
    ax.plot(binned['bin_mid'], p(binned['bin_mid']),
             '--', color='red', alpha=0.5, linewidth=1.5, label='Linear fit')
    
    ax.set_title(col, fontweight='bold', fontsize=10)
    ax.set_xlabel('Nilai Fitur')
    ax.set_ylabel('Churn Rate')
    ax.legend(fontsize=8)

plt.suptitle('Linearitas: Hubungan Fitur vs Churn Rate\n'
              '(garis biru ≠ garis merah = hubungan NON-LINEAR → tree-based lebih cocok)',
              fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../reports/eda_06_linearity_check.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Tersimpan: reports/eda_06_linearity_check.png')

---
## 7. PR-AUC vs AUC-ROC — Metrik yang Tepat untuk Data Imbalanced

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
from imblearn.over_sampling import SMOTE
import joblib

# Load model yang sudah dilatih
model = joblib.load('../models/xgboost_optimized.pkl')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
proba = model.predict_proba(X_test)[:, 1]

auc_roc = roc_auc_score(y_test, proba)
pr_auc  = average_precision_score(y_test, proba)

print(f'AUC-ROC : {auc_roc:.4f}')
print(f'PR-AUC  : {pr_auc:.4f}')
print()
print('Interpretasi:')
print(f'  AUC-ROC {auc_roc:.2f} terlihat "lumayan" padahal data imbalanced')
print(f'  PR-AUC  {pr_auc:.2f} lebih jujur — baseline PR-AUC = {y_test.mean():.2f} (churn rate)')
print(f'  Improvement vs baseline: {(pr_auc - y_test.mean())/y_test.mean()*100:.1f}%')

# Visualisasi
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, proba)
axes[0].plot(fpr, tpr, color=C_NEUTRAL, linewidth=2.5, label=f'XGBoost (AUC={auc_roc:.3f})')
axes[0].plot([0,1],[0,1],'k--', alpha=0.4, label='Random')
axes[0].fill_between(fpr, tpr, alpha=0.1, color=C_NEUTRAL)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve\n(kurang tepat untuk data imbalanced)', fontweight='bold')
axes[0].legend()

# PR Curve
prec_vals, rec_vals, _ = precision_recall_curve(y_test, proba)
axes[1].plot(rec_vals, prec_vals, color=C_CHURN, linewidth=2.5, label=f'XGBoost (PR-AUC={pr_auc:.3f})')
axes[1].axhline(y=y_test.mean(), color='gray', linestyle='--',
                 label=f'Baseline (churn rate={y_test.mean():.2f})')
axes[1].fill_between(rec_vals, prec_vals, alpha=0.1, color=C_CHURN)
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve\n(lebih tepat untuk data imbalanced ✅)', fontweight='bold')
axes[1].legend()

plt.suptitle('AUC-ROC vs PR-AUC — Mana yang Lebih Tepat?', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../reports/eda_07_prauc_vs_auc.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Tersimpan: reports/eda_07_prauc_vs_auc.png')

---
## 8. Ringkasan EDA & Justifikasi Keputusan Modeling

In [ ]:
print('=' * 65)
print('  RINGKASAN EDA & JUSTIFIKASI KEPUTUSAN MODELING')
print('=' * 65)
print()
print('1. CLASS IMBALANCE')
print(f'   Ratio 3.55:1 (tidak churn:churn) → data IMBALANCED')
print(f'   Solusi: SMOTE saat training ✅')
print(f'   Metrik: Gunakan Recall & PR-AUC, bukan hanya AUC-ROC ✅')
print()
print('2. DISTRIBUSI FITUR')
print(f'   Banyak fitur highly skewed (right-skewed)')
print(f'   → Tidak masalah untuk XGBoost/RF ✅')
print(f'   → Akan bermasalah untuk Logistic Regression ❌')
print()
print('3. MULTIKOLINEARITAS')
print(f'   35 pasangan fitur korelasi > 0.85')
print(f'   3 pasangan korelasi = 1.0 (hampir identik)')
print(f'   → Tidak mempengaruhi XGBoost/RF ✅')
print(f'   → Untuk Logistic Regression: WAJIB diatasi ❌')
print(f'   → Untuk SHAP: sebaiknya drop fitur duplikat')
print()
print('4. LINEARITAS')
print(f'   Hubungan fitur-churn mayoritas NON-LINEAR')
print(f'   → Mendukung pilihan XGBoost & Random Forest ✅')
print(f'   → Logistic Regression tidak cocok ❌')
print()
print('5. METRIK EVALUASI YANG TEPAT')
print(f'   ✅ Recall         — prioritas bisnis (minimasi churn terlewat)')
print(f'   ✅ PR-AUC         — tepat untuk data imbalanced')
print(f'   ⚠️  AUC-ROC       — kurang tepat, bisa menyesatkan')
print(f'   ⚠️  Accuracy      — sangat menyesatkan untuk imbalanced data')
print()
print('KESIMPULAN:')
print('  Semua temuan EDA mendukung keputusan menggunakan')
print('  XGBoost & Random Forest sebagai model utama.')